# A CUDA kernel from a `%%rust` cell

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xavierforge/colab-rust/blob/main/examples/03_cudarc_gpu.ipynb)

This notebook needs a GPU runtime (Runtime, Change runtime type, T4). It compiles a CUDA kernel with nvrtc and launches it through [cudarc](https://github.com/coreylowman/cudarc), all inside `%%rust` cells on Colab's Python kernel.


## Check the GPU

The last line loads the two libraries cudarc will open at run time (`libcuda` from the driver, `libnvrtc` from the toolkit) the same way cudarc does, by name. Nothing links against CUDA at build time.


In [1]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -1
import ctypes
for lib in ("libcuda.so.1", "libnvrtc.so"):
    ctypes.CDLL(lib)
    print(f"{lib} loads")

name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB
Build cuda_12.8.r12.8/compiler.35583870_0
libcuda.so.1 loads
libnvrtc.so loads


## Set up colab-rust


In [2]:
!curl -fsSL -o setup.sh https://raw.githubusercontent.com/xavierforge/colab-rust/main/setup.sh
!bash setup.sh
%load_ext colab_rust

▶ colab-rust setup (ref: main)
▶ Ubuntu 24.04, glibc 2.39 (prebuilt: Ubuntu 22.04, needs glibc >= 2.35)
▶ Installing Rust (stable, minimal profile)...
✅ Rust 1.98.1
✅ evcxr_jupyter at /root/.cargo/bin/evcxr_jupyter
✅ Rust kernel registered with Jupyter
▶ Installing colab_rust.py magic module...
✅ colab_rust.py 0.1.4 ready at /content/colab_rust.py (ref: main)

🎉 Setup complete.

Next steps in your notebook:

    %load_ext colab_rust

    %%rust
    println!("Hello from Rust on Colab!");

For compile-heavy crates (candle, tch), prefer a Cargo project
with !cargo run over :dep. See examples/ for patterns.
✅ colab_rust 0.1.4 loaded — use %%rust in any cell


## Add cudarc

`dynamic-loading` means the crate looks up `libcuda` and `libnvrtc` when first used instead of linking them, `cuda-12040` picks the API level (any newer toolkit is fine), and `std` lets `?` work on cudarc's errors. This is the only cell that compiles anything substantial; the progress line under it is the cold build time.


In [3]:
%%rust
:dep cudarc = { version = "0.19", default-features = false, features = ["driver", "nvrtc", "dynamic-loading", "cuda-12040", "std"] }
println!("cudarc ready");

   Compiled 2 crates in 11s

cudarc ready


## Vector add on the GPU

The kernel is plain CUDA C in a string. nvrtc compiles it to PTX at run time, the driver loads the module, and the launch builder passes the arguments. One million elements go up, get added, and come back.


In [4]:
%%rust
use cudarc::driver::{CudaContext, LaunchConfig, PushKernelArg};
use cudarc::nvrtc::compile_ptx;

const VEC_ADD: &str = r#"
extern "C" __global__ void vec_add(const float* a, const float* b, float* out, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) { out[i] = a[i] + b[i]; }
}
"#;

let ctx = CudaContext::new(0)?;
println!("GPU: {}", ctx.name()?);
let stream = ctx.default_stream();

let module = ctx.load_module(compile_ptx(VEC_ADD)?)?;
let vec_add = module.load_function("vec_add")?;

let n = 1 << 20;
let a: Vec<f32> = (0..n).map(|i| i as f32).collect();
let b: Vec<f32> = (0..n).map(|i| 2.0 * i as f32).collect();
let a_dev = stream.clone_htod(&a)?;
let b_dev = stream.clone_htod(&b)?;
let mut out_dev = stream.alloc_zeros::<f32>(n)?;

let n_i32 = n as i32;
// Build and launch in one expression: the launch builder borrows the buffers,
// and evcxr can only keep variables that own their data between cells.
unsafe {
    stream.launch_builder(&vec_add)
        .arg(&a_dev).arg(&b_dev).arg(&mut out_dev).arg(&n_i32)
        .launch(LaunchConfig::for_num_elems(n as u32))?
};

let out = stream.clone_dtoh(&out_dev)?;
let ok = out.iter().zip(a.iter().zip(&b)).all(|(o, (x, y))| *o == x + y);
println!("out[0..5] = {:?}", &out[..5]);
println!("{n} elements, all correct: {ok}");

GPU: Tesla T4
out[0..5] = [0.0, 3.0, 6.0, 9.0, 12.0]
1048576 elements, all correct: true


## What happened

Everything ran in evcxr's runtime process, which colab-rust started as a subprocess of Colab's Python kernel. The GPU is the same one Python sees, so a Rust cell can hand data to a Python cell and back. On a CPU runtime `CudaContext::new` fails with a clear error about `libcuda`, which is the expected way to find out you forgot to switch the runtime.
